# Bước C · PCD và baseline Qwen
Chạy các nhóm đã chọn với Front/Behind gốc. Kết quả hai nhánh đối chứng được lưu riêng. Chạy Run All với T4 ×2 và Internet.

## 1. Cài các thư viện còn thiếu

Giữ PyTorch/torchvision có sẵn của Kaggle, dùng Transformers 4.51.3 theo repo.
Không cài `requirements.txt` gốc vì sẽ thay cả stack torch/numpy/pandas của Kaggle.
Nếu đã import Transformers trước khi chạy cell này, restart session rồi chạy từ đầu.
Không cần `flash-attn`, `bitsandbytes`, `datasets` hoặc Hugging Face token cho model public này.

In [ ]:
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

# Keep Kaggle's matching torch/torchvision/CUDA installation.
# Run this cell before importing transformers; restart the session if upgrading an imported copy.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
packages = [
    "transformers==4.51.3", "accelerate==1.6.0", "huggingface_hub==0.30.2",
    "tokenizers==0.21.1", "safetensors>=0.4.3,<1", "pandas>=2.2,<3.1",
    "Pillow>=10,<13", "tqdm>=4.66,<5", "openpyxl>=3.1,<4",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "--upgrade-strategy", "only-if-needed", *packages])
print({name: importlib.metadata.version(name)
       for name in ["torch", "torchvision", "transformers", "accelerate", "huggingface_hub"]})

## 2. Cấu hình

Lần đầu giữ mặc định. Có thể gắn model/TSV từ Kaggle Inputs để tránh tải lại.

In [ ]:
from pathlib import Path

MODE = "full"
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
MODEL_REVISION = "66285546d2b821cf421d4f5eb2576359d3770cd3"
DATASET_ID = "ccvl/3DSRBench"
DATASET_REVISION = "5c90e422b68d0484d50a3ff06b3096ea7d288026"
TSV_FILENAME = "3dsrbench_v1_vlmevalkit_circular.tsv"
ALPHA = 4.0
NUM_WORKERS = 2
SEED = 42
COMPUTE_BASELINE = True  # Original allocentric prompt, greedy decoding; extra inference cost.

# None preserves the repository's default AutoProcessor image resolution.
# For a lower-memory experiment, set MAX_PIXELS = 1280 * 28 * 28 (recorded in manifest).
# Changing this setting can change accuracy; use a new OUTPUT_DIR.
MIN_PIXELS = None
MAX_PIXELS = None

# Optional mounted Kaggle inputs. None tries /kaggle/input, then downloads the public files.
# MODEL_DIR must contain config, tokenizer, processor, and ALL safetensors shards.
MODEL_DIR = None  # e.g. "/kaggle/input/qwen25vl3b/transformers/default/1"
TSV_PATH = None   # e.g. "/kaggle/input/3dsrbench/3dsrbench_v1_vlmevalkit_circular.tsv"
SANITY_QID = None  # None chooses the first supported row; exact qid selects another image.

OUTPUT_DIR = Path("/kaggle/working/pcd_step_c_replay")
# Same session: rerun the worker cell. New session: attach the old output as Kaggle input,
# point RESUME_FROM at the directory containing run_manifest.json + predictions_gpu*.jsonl.
RESUME_FROM = None
assert ALPHA >= 0 and NUM_WORKERS == 2
if MIN_PIXELS is not None and MAX_PIXELS is not None:
    assert 0 < MIN_PIXELS <= MAX_PIXELS
OLD={'04B71HRN-flip': {'id': '04B71HRN-flip', 'group': '04B71HRN-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, '04B71HRN-flip-1': {'id': '04B71HRN-flip-1', 'group': '04B71HRN-flip', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, '089B1CW6': {'id': '089B1CW6', 'group': '089B1CW6', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, '089B1CW6-1': {'id': '089B1CW6-1', 'group': '089B1CW6', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, '0UZR0T93-flip': {'id': '0UZR0T93-flip', 'group': '0UZR0T93-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, '0UZR0T93-flip-1': {'id': '0UZR0T93-flip-1', 'group': '0UZR0T93-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '0XIZ0MZC': {'id': '0XIZ0MZC', 'group': '0XIZ0MZC', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '0XIZ0MZC-1': {'id': '0XIZ0MZC-1', 'group': '0XIZ0MZC', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '17BXE5S2-flip': {'id': '17BXE5S2-flip', 'group': '17BXE5S2-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, '17BXE5S2-flip-1': {'id': '17BXE5S2-flip-1', 'group': '17BXE5S2-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, '1INS9OIW': {'id': '1INS9OIW', 'group': '1INS9OIW', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, '1INS9OIW-1': {'id': '1INS9OIW-1', 'group': '1INS9OIW', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, '1ZLFEF7B': {'id': '1ZLFEF7B', 'group': '1ZLFEF7B', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '1ZLFEF7B-1': {'id': '1ZLFEF7B-1', 'group': '1ZLFEF7B', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '3N009IX6-flip': {'id': '3N009IX6-flip', 'group': '3N009IX6-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '3N009IX6-flip-1': {'id': '3N009IX6-flip-1', 'group': '3N009IX6-flip', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '40ZZU3JS': {'id': '40ZZU3JS', 'group': '40ZZU3JS', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, '40ZZU3JS-1': {'id': '40ZZU3JS-1', 'group': '40ZZU3JS', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, '4HHN5X2O': {'id': '4HHN5X2O', 'group': '4HHN5X2O', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, '4HHN5X2O-1': {'id': '4HHN5X2O-1', 'group': '4HHN5X2O', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, '7N2PSCLH': {'id': '7N2PSCLH', 'group': '7N2PSCLH', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, '7N2PSCLH-1': {'id': '7N2PSCLH-1', 'group': '7N2PSCLH', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, '7UFJG2VT': {'id': '7UFJG2VT', 'group': '7UFJG2VT', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, '7UFJG2VT-1': {'id': '7UFJG2VT-1', 'group': '7UFJG2VT', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, '7ZGPUU71': {'id': '7ZGPUU71', 'group': '7ZGPUU71', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '7ZGPUU71-1': {'id': '7ZGPUU71-1', 'group': '7ZGPUU71', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '81572ZQB': {'id': '81572ZQB', 'group': '81572ZQB', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '81572ZQB-1': {'id': '81572ZQB-1', 'group': '81572ZQB', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, '93O1RXPD-flip': {'id': '93O1RXPD-flip', 'group': '93O1RXPD-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, '93O1RXPD-flip-1': {'id': '93O1RXPD-flip-1', 'group': '93O1RXPD-flip', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, '94AHRB8A-flip': {'id': '94AHRB8A-flip', 'group': '94AHRB8A-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, '94AHRB8A-flip-1': {'id': '94AHRB8A-flip-1', 'group': '94AHRB8A-flip', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'B', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'A2GTCTDC': {'id': 'A2GTCTDC', 'group': 'A2GTCTDC', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'A2GTCTDC-1': {'id': 'A2GTCTDC-1', 'group': 'A2GTCTDC', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'A2GTCTDC-flip': {'id': 'A2GTCTDC-flip', 'group': 'A2GTCTDC-flip', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'A2GTCTDC-flip-1': {'id': 'A2GTCTDC-flip-1', 'group': 'A2GTCTDC-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'AFHNGOS7': {'id': 'AFHNGOS7', 'group': 'AFHNGOS7', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'AFHNGOS7-1': {'id': 'AFHNGOS7-1', 'group': 'AFHNGOS7', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'AMQKZNEO': {'id': 'AMQKZNEO', 'group': 'AMQKZNEO', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'AMQKZNEO-1': {'id': 'AMQKZNEO-1', 'group': 'AMQKZNEO', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'C718L2MO': {'id': 'C718L2MO', 'group': 'C718L2MO', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'C718L2MO-1': {'id': 'C718L2MO-1', 'group': 'C718L2MO', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'D4ZILX77-flip': {'id': 'D4ZILX77-flip', 'group': 'D4ZILX77-flip', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'D4ZILX77-flip-1': {'id': 'D4ZILX77-flip-1', 'group': 'D4ZILX77-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'E0TMY1C2-flip': {'id': 'E0TMY1C2-flip', 'group': 'E0TMY1C2-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'E0TMY1C2-flip-1': {'id': 'E0TMY1C2-flip-1', 'group': 'E0TMY1C2-flip', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'EC1RRHHK': {'id': 'EC1RRHHK', 'group': 'EC1RRHHK', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'EC1RRHHK-1': {'id': 'EC1RRHHK-1', 'group': 'EC1RRHHK', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'EC96SY1L': {'id': 'EC96SY1L', 'group': 'EC96SY1L', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'EC96SY1L-1': {'id': 'EC96SY1L-1', 'group': 'EC96SY1L', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'EEKA78V5-flip': {'id': 'EEKA78V5-flip', 'group': 'EEKA78V5-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'EEKA78V5-flip-1': {'id': 'EEKA78V5-flip-1', 'group': 'EEKA78V5-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'FWD4EDQZ-flip': {'id': 'FWD4EDQZ-flip', 'group': 'FWD4EDQZ-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'FWD4EDQZ-flip-1': {'id': 'FWD4EDQZ-flip-1', 'group': 'FWD4EDQZ-flip', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'GKOMX64F': {'id': 'GKOMX64F', 'group': 'GKOMX64F', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'GKOMX64F-1': {'id': 'GKOMX64F-1', 'group': 'GKOMX64F', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'A', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'GM6UPREN-flip': {'id': 'GM6UPREN-flip', 'group': 'GM6UPREN-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'GM6UPREN-flip-1': {'id': 'GM6UPREN-flip-1', 'group': 'GM6UPREN-flip', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'HOWNMJFN': {'id': 'HOWNMJFN', 'group': 'HOWNMJFN', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'HOWNMJFN-1': {'id': 'HOWNMJFN-1', 'group': 'HOWNMJFN', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'I5C54XA1-flip': {'id': 'I5C54XA1-flip', 'group': 'I5C54XA1-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'I5C54XA1-flip-1': {'id': 'I5C54XA1-flip-1', 'group': 'I5C54XA1-flip', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'J1I9ICJR-flip': {'id': 'J1I9ICJR-flip', 'group': 'J1I9ICJR-flip', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'J1I9ICJR-flip-1': {'id': 'J1I9ICJR-flip-1', 'group': 'J1I9ICJR-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'MC85NOOX': {'id': 'MC85NOOX', 'group': 'MC85NOOX', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'MC85NOOX-1': {'id': 'MC85NOOX-1', 'group': 'MC85NOOX', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'MNZJ0633-flip': {'id': 'MNZJ0633-flip', 'group': 'MNZJ0633-flip', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'MNZJ0633-flip-1': {'id': 'MNZJ0633-flip-1', 'group': 'MNZJ0633-flip', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'O3TTEOW6-flip': {'id': 'O3TTEOW6-flip', 'group': 'O3TTEOW6-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'O3TTEOW6-flip-1': {'id': 'O3TTEOW6-flip-1', 'group': 'O3TTEOW6-flip', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'A', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'QTGS92OS-flip': {'id': 'QTGS92OS-flip', 'group': 'QTGS92OS-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'QTGS92OS-flip-1': {'id': 'QTGS92OS-flip-1', 'group': 'QTGS92OS-flip', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'R15IP3M1': {'id': 'R15IP3M1', 'group': 'R15IP3M1', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'R15IP3M1-1': {'id': 'R15IP3M1-1', 'group': 'R15IP3M1', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'R3RTXLVO-flip': {'id': 'R3RTXLVO-flip', 'group': 'R3RTXLVO-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'R3RTXLVO-flip-1': {'id': 'R3RTXLVO-flip-1', 'group': 'R3RTXLVO-flip', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'R4BJUZ7T': {'id': 'R4BJUZ7T', 'group': 'R4BJUZ7T', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'R4BJUZ7T-1': {'id': 'R4BJUZ7T-1', 'group': 'R4BJUZ7T', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': False, 'baseline_hit': True, 'status': 'ok'}, 'S940WKSC': {'id': 'S940WKSC', 'group': 'S940WKSC', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'S940WKSC-1': {'id': 'S940WKSC-1', 'group': 'S940WKSC', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'UFY46JSW': {'id': 'UFY46JSW', 'group': 'UFY46JSW', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'B', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'UFY46JSW-1': {'id': 'UFY46JSW-1', 'group': 'UFY46JSW', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'UIN35VUM': {'id': 'UIN35VUM', 'group': 'UIN35VUM', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'UIN35VUM-1': {'id': 'UIN35VUM-1', 'group': 'UIN35VUM', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'USC0VVQ8': {'id': 'USC0VVQ8', 'group': 'USC0VVQ8', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'USC0VVQ8-1': {'id': 'USC0VVQ8-1', 'group': 'USC0VVQ8', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'X33PI17P-flip': {'id': 'X33PI17P-flip', 'group': 'X33PI17P-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'X33PI17P-flip-1': {'id': 'X33PI17P-flip-1', 'group': 'X33PI17P-flip', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'Y9CPSAKW-flip': {'id': 'Y9CPSAKW-flip', 'group': 'Y9CPSAKW-flip', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'A', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'Y9CPSAKW-flip-1': {'id': 'Y9CPSAKW-flip-1', 'group': 'Y9CPSAKW-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': False, 'baseline_hit': False, 'status': 'ok'}, 'YEMQRF30-flip': {'id': 'YEMQRF30-flip', 'group': 'YEMQRF30-flip', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'B', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'YEMQRF30-flip-1': {'id': 'YEMQRF30-flip-1', 'group': 'YEMQRF30-flip', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'A', 'hit': True, 'baseline_hit': True, 'status': 'ok'}, 'YK1SIQ00': {'id': 'YK1SIQ00', 'group': 'YK1SIQ00', 'task': 'orientation_on_the_left', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'YK1SIQ00-1': {'id': 'YK1SIQ00-1', 'group': 'YK1SIQ00', 'task': 'orientation_on_the_left', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'ZBLMPXID-flip': {'id': 'ZBLMPXID-flip', 'group': 'ZBLMPXID-flip', 'task': 'orientation_in_front_of', 'prediction': 'A', 'baseline': 'B', 'hit': True, 'baseline_hit': False, 'status': 'ok'}, 'ZBLMPXID-flip-1': {'id': 'ZBLMPXID-flip-1', 'group': 'ZBLMPXID-flip', 'task': 'orientation_in_front_of', 'prediction': 'B', 'baseline': 'A', 'hit': True, 'baseline_hit': False, 'status': 'ok'}}

## 3. Kiểm tra 2 GPU và chuẩn bị dữ liệu/model

TSV khoảng 843 MB; model cần thêm vài GB dung lượng tải. Tải một lần, cả hai worker dùng chung cache.

In [ ]:
import json
import torch
from huggingface_hub import hf_hub_download, snapshot_download

assert torch.cuda.is_available(), "Chọn Settings → Accelerator → GPU T4 x2."
assert torch.cuda.device_count() == 2, f"Cần đúng 2 GPU, thấy {torch.cuda.device_count()}."
for gpu in range(2):
    info = torch.cuda.get_device_properties(gpu)
    print(f"GPU {gpu}: {info.name}, {info.total_memory / 2**30:.1f} GiB")
print("Workers use FP16 + SDPA; each worker loads a full 3B model on its own GPU.")

input_root = Path("/kaggle/input")
if TSV_PATH is None:
    candidates = sorted(input_root.rglob(TSV_FILENAME)) if input_root.exists() else []
    if len(candidates) > 1:
        raise ValueError("Nhiều TSV được tìm thấy; đặt TSV_PATH rõ ràng.")
    if candidates:
        TSV_PATH = str(candidates[0])
    else:
        TSV_PATH = hf_hub_download(repo_id=DATASET_ID, filename=TSV_FILENAME,
                                   repo_type="dataset", revision=DATASET_REVISION)
if MODEL_DIR is None:
    candidates = []
    for path in input_root.rglob("config.json") if input_root.exists() else []:
        try:
            metadata = json.loads(path.read_text())
        except (ValueError, OSError):
            continue
        if (metadata.get("model_type") == "qwen2_5_vl"
                and metadata.get("hidden_size") == 2048
                and metadata.get("num_hidden_layers") == 36
                and list(path.parent.glob("*.safetensors"))):
            candidates.append(path.parent)
    if len(candidates) > 1:
        raise ValueError("Nhiều bản model được tìm thấy; đặt MODEL_DIR rõ ràng.")
    if candidates:
        MODEL_DIR = str(candidates[0])
    else:
        MODEL_DIR = snapshot_download(
            MODEL_ID, revision=MODEL_REVISION,
            allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"],
            max_workers=2,
        )
model_path = Path(MODEL_DIR).resolve()
model_metadata = json.loads((model_path / "config.json").read_text())
assert (model_metadata.get("model_type"), model_metadata.get("hidden_size"),
        model_metadata.get("num_hidden_layers")) == ("qwen2_5_vl", 2048, 36), "Sai model 3B."
assert (model_path / "preprocessor_config.json").is_file(), "Thiếu processor."
assert (model_path / "tokenizer_config.json").is_file(), "Thiếu tokenizer."
index_path = model_path / "model.safetensors.index.json"
if index_path.exists():
    weight_names = set(json.loads(index_path.read_text())["weight_map"].values())
    assert all((model_path / name).is_file() for name in weight_names), "Thiếu weight shard."
else:
    assert (model_path / "model.safetensors").is_file(), "Thiếu model.safetensors hoặc shard index."
assert Path(TSV_PATH).is_file(), f"Không tìm thấy TSV: {TSV_PATH}"
print("Model directory:", model_path)
print("Circular TSV:", TSV_PATH)

## 4. Mã PCD gốc và bộ điều phối

Hai cell tiếp theo ghi module vào `/kaggle/working`. Mã gốc được nhúng nguyên văn;
bộ điều phối chỉ bổ sung chia GPU, xử lý dữ liệu, cache và xuất kết quả.
Mỗi tiến trình chỉ thấy một GPU, dùng FP16 + SDPA, greedy decoding.

In [ ]:
%%writefile /kaggle/working/pcd_reference.py
import os
import json
import argparse
extraction_trace = {}
import base64
import re
from io import BytesIO

import torch
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from tqdm import tqdm

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor


def parse_args():
    parser = argparse.ArgumentParser(
        description="Run PCD on 3DSRBench orientation tasks"
    )
    parser.add_argument(
        "--model_size",
        type=str,
        required=True,
        choices=["3B", "7B", "32B"],
        help="Model size to use (3B, 7B, or 32B)"
    )
    parser.add_argument(
        "--alpha",
        type=float,
        default=4.0,
        help="Alpha parameter for contrastive decoding"
    )
    parser.add_argument(
        "--data_path",
        type=str,
        default="../../data/3dsrbench_v1_vlmevalkit_circular.tsv",
        help="Path to the TSV data file"
    )
    parser.add_argument(
        "--output_root",
        type=str,
        default="../../outputs/3dsrbench",
        help="Root directory for outputs"
    )
    return parser.parse_args()


def decode_image(base64_str):
    image_data = base64.b64decode(base64_str)
    return Image.open(BytesIO(image_data)).convert("RGB")


def is_valid_option(x):
    if x is None:
        return False
    if isinstance(x, float) and pd.isna(x):
        return False
    return str(x).strip() != ""


def build_options(row):
    options = {}
    for k in ["A", "B", "C", "D"]:
        if k in row and is_valid_option(row[k]):
            options[k] = str(row[k]).strip()
    return options, list(options.keys())


def options_to_text(options_dict):
    return "\n".join([f"{k}. {v}" for k, v in options_dict.items()])


LEFT_RIGHT_EXTRACTION_PROMPT = """\
Extract the "reference object" and "target object" from a spatial reasoning question.
- reference object: the object whose position and facing direction defines the viewpoint
- target object: the object whose left/right position is being asked about

Return your answer strictly in this JSON format (no extra text):
{"ref_obj": "<reference object>", "target_obj": "<target object>"}

Examples:

Q: Consider the real-world 3D locations and orientations of the objects. If I stand at the chair's position facing where it is facing, is the desk on the left or right of me?
A: {"ref_obj": "chair", "target_obj": "desk"}

Q: Consider the real-world 3D locations and orientations of the objects. If I stand at the green car's position facing where it is facing, is the traffic cone on the left or right of me?
A: {"ref_obj": "green car", "target_obj": "traffic cone"}

Q: Consider the real-world 3D locations and orientations of the objects. If I stand at the person's position facing where it is facing, is the bicycle on the left or right of me?
A: {"ref_obj": "person", "target_obj": "bicycle"}

Q: Consider the real-world 3D locations and orientations of the objects. If I stand at the woman in pink cloth's position facing where it is facing, is the man holding a bag on the left or right of me?
A: {"ref_obj": "woman in pink cloth", "target_obj": "man holding a bag"}

Q: {question}
A:"""


FRONT_BEHIND_EXTRACTION_PROMPT = """\
Extract the "reference object" and "target object" from a spatial reasoning question.
- reference object: the object whose position and facing direction defines the viewpoint
- target object: the object whose front/behind position is being asked about
Return your answer strictly in this JSON format (no extra text):
{"ref_obj": "<reference object>", "target_obj": "<target object>"}

Examples:
Q: Consider the real-world 3D locations and orientations of the objects. If I stand at the table's position facing where it is facing, is the bookshelf in front of me or behind me?
A: {"ref_obj": "table", "target_obj": "bookshelf"}

Q: Consider the real-world 3D locations and orientations of the objects. If I stand at the truck's position facing where it is facing, is the gray car in front of me or behind me?
A: {"ref_obj": "truck", "target_obj": "stop sign"}

Q: Consider the real-world 3D locations and orientations of the objects. If I stand at the man's position facing where it is facing, is the backpack in front of me or behind me?
A: {"ref_obj": "man", "target_obj": "gray car"}

Q: Consider the real-world 3D locations and orientations of the objects. If I stand at the girl in red dress's position facing where it is facing, is the boy with glasses in front of me or behind me?
A: {"ref_obj": "girl in red dress", "target_obj": "boy with glasses"}

Q: {question}
A:"""


@torch.no_grad()
def extract_objects_via_llm(question, category, model, processor, device):
    if category == "orientation_on_the_left":
        prompt_template = LEFT_RIGHT_EXTRACTION_PROMPT
    else:
        prompt_template = FRONT_BEHIND_EXTRACTION_PROMPT
    
    prompt = prompt_template.replace("{question}", question.strip())

    messages = [{
        "role": "user",
        "content": [{"type": "text", "text": prompt}],
    }]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        padding=True,
        return_tensors="pt"
    )
    for k, v in inputs.items():
        if torch.is_tensor(v):
            inputs[k] = v.to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
    )

    generated_text = processor.batch_decode(
        outputs[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )[0].strip()

    try:
        extraction_trace[category,question]=(prompt,generated_text)
        json_match = re.search(r'\{[^{}]*\}', generated_text)
        if json_match:
            parsed = json.loads(json_match.group())
            ref_obj = str(parsed.get("ref_obj", "object")).strip()
            target_obj = str(parsed.get("target_obj", "object")).strip()
            if ref_obj and target_obj:
                return ref_obj, target_obj
    except (json.JSONDecodeError, KeyError, TypeError):
        pass

    return "object", "object"


def build_rewrite_q(ref_obj, target_obj, category):
    if category == "orientation_on_the_left":
        return f"Is the {target_obj} on the left or right of the {ref_obj}?"
    else:
        return f"is the {target_obj} in front of the {ref_obj} or behind the {ref_obj}?"


def extract_question_body(question):
    prefix_patterns = [
        "Consider the real-world 3D locations and orientations of the objects. ",
        "Consider the real-world 3D locations and orientations of the objects.\n",
    ]
    
    question_body = question
    for pattern in prefix_patterns:
        if question.startswith(pattern):
            question_body = question[len(pattern):]
            break
    
    return question_body


def build_original_prompt(question, options_text):
    prefix = "Consider the real-world 3D locations and orientations of the objects."
    question_body = extract_question_body(question)
    return f"""Question: {prefix} {question_body}

Options:
{options_text}

Return only the correct option letter.
"""


def build_camera_prompt(rewrite_q, options_text):
    prefix = "Consider the real-world 3D locations and orientations of the objects."
    return f"""Question: {prefix} From the camera view, {rewrite_q}

Options:
{options_text}

Return only the correct option letter.
"""


def build_viewpoint_check_prompt(ref_obj, category):
    if category == "orientation_on_the_left":
        return (
            f"Consider the real-world 3D locations and orientations of the objects.\n"
            f"Question:\n"
            f"In image, are you viewing the {ref_obj}'s back side?\n"
            f"Answer only yes or no."
        )
    else:
        return (
            f"Question: Consider the real-world 3D locations and orientations of the objects."
            f"In image, are you looking at the {ref_obj} from behind?"
            f"Answer only yes or no."
        )


def build_model_inputs(image, prompt, processor, device):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ],
    }]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    )

    for k, v in inputs.items():
        if torch.is_tensor(v):
            inputs[k] = v.to(device)

    return inputs


@torch.no_grad()
def generate_answer(image, prompt, model, processor, device):
    inputs = build_model_inputs(image, prompt, processor, device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

    generated_text = processor.batch_decode(
        outputs[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )[0].strip()

    return generated_text


@torch.no_grad()
def check_same_viewpoint(image, ref_obj, category, model, processor, device):
    prompt = build_viewpoint_check_prompt(ref_obj, category)
    raw = generate_answer(image, prompt, model, processor, device)
    
    if category == "orientation_on_the_left":
        is_same = raw.lower().startswith("yes")
    else:
        is_same = raw.lower().startswith("no")
    
    return raw, is_same


def get_candidate_token_ids_for_label(label, tokenizer):
    variants = [label, " " + label, "\n" + label]

    ids = set()
    for v in variants:
        tok = tokenizer.encode(v, add_special_tokens=False)
        if len(tok) == 1:
            ids.add(tok[0])

    if not ids:
        tok = tokenizer.encode(label, add_special_tokens=False)
        ids.add(tok[-1])

    return sorted(ids)


@torch.no_grad()
def score_next_token_logprobs(image, prompt, model, processor, device):
    inputs = build_model_inputs(image, prompt, processor, device)
    outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    logp = F.log_softmax(logits, dim=-1)[0]
    return logp


def contrastive_pick_option(image, original_prompt, camera_prompt, allowed_labels, 
                            alpha, model, processor, device, option_token_ids):
    orig_logp = score_next_token_logprobs(image, original_prompt, model, processor, device)
    cam_logp = score_next_token_logprobs(image, camera_prompt, model, processor, device)

    scores = {}
    for label in allowed_labels:
        ids = option_token_ids[label]

        orig_best = max(orig_logp[i].item() for i in ids)
        cam_best = max(cam_logp[i].item() for i in ids)

        scores[label] = {
            "orig": orig_best,
            "cam": cam_best,
            "contrastive": orig_best - alpha * cam_best
        }

    pred = max(scores.items(), key=lambda x: x[1]["contrastive"])[0]
    return pred, scores


def process_category(df_category, category, model, processor, device, alpha, 
                     model_name_safe, output_dir, option_token_ids):
    results = []
    logs = []
    
    extraction_cache_path = os.path.join(
        output_dir,
        f"{model_name_safe}_{category}_extraction_cache.json"
    )
    
    viewpoint_cache_path = os.path.join(
        output_dir,
        f"{model_name_safe}_{category}_viewpoint_cache.json"
    )
    
    if os.path.exists(extraction_cache_path):
        with open(extraction_cache_path, "r") as f:
            extraction_cache = json.load(f)
    else:
        extraction_cache = {}
    
    if os.path.exists(viewpoint_cache_path):
        with open(viewpoint_cache_path, "r") as f:
            viewpoint_cache = json.load(f)
    else:
        viewpoint_cache = {}
    
    for idx in tqdm(range(len(df_category)), desc=f"Processing {category}"):
        row = df_category.iloc[idx]
        qid = str(row["qid"])
        question = str(row["question"])

        image = decode_image(row["image"])
        options_dict, labels = build_options(row)
        options_text = options_to_text(options_dict)

        if qid in extraction_cache:
            ref_obj = extraction_cache[qid]["ref_obj"]
            target_obj = extraction_cache[qid]["target_obj"]
        else:
            ref_obj, target_obj = extract_objects_via_llm(
                question, category, model, processor, device
            )
            extraction_cache[qid] = {"ref_obj": ref_obj, "target_obj": target_obj}
            with open(extraction_cache_path, "w") as f:
                json.dump(extraction_cache, f, indent=2, ensure_ascii=False)

        rewrite_q = build_rewrite_q(ref_obj, target_obj, category)
        
        original_prompt = build_original_prompt(question, options_text)
        camera_prompt = build_camera_prompt(rewrite_q, options_text)

        if qid in viewpoint_cache:
            vp_raw = viewpoint_cache[qid]["raw"]
            is_same_vp = viewpoint_cache[qid]["is_same"]
        else:
            vp_raw, is_same_vp = check_same_viewpoint(
                image, ref_obj, category, model, processor, device
            )
            viewpoint_cache[qid] = {"raw": vp_raw, "is_same": is_same_vp}
            with open(viewpoint_cache_path, "w") as f:
                json.dump(viewpoint_cache, f, indent=2, ensure_ascii=False)

        if is_same_vp:
            pred = generate_answer(image, camera_prompt, model, processor, device)
            pred_letter = pred.strip()[0].upper() if pred.strip() else "A"
            if pred_letter not in labels:
                pred_letter = labels[0]
            pred = pred_letter
        else:
            pred, _ = contrastive_pick_option(
                image=image,
                original_prompt=original_prompt,
                camera_prompt=camera_prompt,
                allowed_labels=labels,
                alpha=alpha,
                model=model,
                processor=processor,
                device=device,
                option_token_ids=option_token_ids
            )

        answer = str(row["answer"]).strip()
        hit = int(pred == answer)

        results.append([
            idx,
            str(row["qid"]),
            str(row["question"]),
            options_dict.get("A", ""),
            options_dict.get("B", ""),
            options_dict.get("C", ""),
            options_dict.get("D", ""),
            pred,
            str(row["category"]),
            rewrite_q,
            pred,
            answer,
            hit
        ])

        log_entry = {
            "qid": qid,
            "category": str(row["category"]),
            "question": question,
            "prediction": pred,
            "answer": answer,
            "hit": hit
        }
        
        logs.append(log_entry)
    
    with open(extraction_cache_path, "w") as f:
        json.dump(extraction_cache, f, indent=2, ensure_ascii=False)
    
    with open(viewpoint_cache_path, "w") as f:
        json.dump(viewpoint_cache, f, indent=2, ensure_ascii=False)
    
    return results, logs


def main():
    args = parse_args()
    
    model_size_map = {
        "3B": "Qwen/Qwen2.5-VL-3B-Instruct",
        "7B": "Qwen/Qwen2.5-VL-7B-Instruct",
        "32B": "Qwen/Qwen2.5-VL-32B-Instruct"
    }
    
    MODEL_NAME = model_size_map[args.model_size]
    
    alpha_str = str(args.alpha).replace(".", "p")
    model_name_safe = f"PCD_Qwen2.5-VL-{args.model_size}-Instruct_alpha{alpha_str}"
    
    OUTPUT_DIR = os.path.join(args.output_root, model_name_safe)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    print(f"\n{'='*60}")
    print(f"Model: {MODEL_NAME}")
    print(f"Alpha: {args.alpha}")
    print(f"Output Directory: {OUTPUT_DIR}")
    print(f"Device: {device}")
    print(f"{'='*60}\n")
    
    print("Loading model...")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    model.eval()
    
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    tokenizer = processor.tokenizer
    print("Model loaded successfully!\n")
    
    option_token_ids = {
        label: get_candidate_token_ids_for_label(label, tokenizer)
        for label in ["A", "B", "C", "D"]
    }
    
    print(f"Loading data from {args.data_path}...")
    df = pd.read_csv(args.data_path, sep="\t")
    
    categories = ["orientation_on_the_left", "orientation_in_front_of"]
    df_filtered = df[df["category"].isin(categories)].reset_index(drop=True)
    
    print(f"Total samples: {len(df_filtered)}")
    for cat in categories:
        count = len(df_filtered[df_filtered["category"] == cat])
        print(f"  - {cat}: {count}")
    print()
    
    all_results = []
    all_logs = []
    
    for category in categories:
        print(f"\nProcessing category: {category}")
        
        df_category = df_filtered[df_filtered["category"] == category].reset_index(drop=True)
        results, logs = process_category(
            df_category=df_category,
            category=category,
            model=model,
            processor=processor,
            device=device,
            alpha=args.alpha,
            model_name_safe=model_name_safe,
            output_dir=OUTPUT_DIR,
            option_token_ids=option_token_ids
        )
        
        all_results.extend(results)
        all_logs.extend(logs)
        
        cat_acc = sum(log["hit"] for log in logs) / len(logs) if logs else 0
        print(f"Accuracy: {cat_acc:.4f}")
    
    excel_path = os.path.join(
        OUTPUT_DIR,
        f"{model_name_safe}_3DSRBenchv1_openai_result.xlsx"
    )
    
    json_path = os.path.join(
        OUTPUT_DIR,
        f"{model_name_safe}_results.json"
    )
    
    for i, result in enumerate(all_results):
        result[0] = i
    
    pd.DataFrame(all_results, columns=[
        "index", "qid", "question", "A", "B", "C", "D",
        "prediction", "category", "col9", "col10", "answer", "hit"
    ]).to_excel(excel_path, index=False)
    
    with open(json_path, "w") as f:
        json.dump(all_logs, f, indent=2, ensure_ascii=False)
    
    total = len(all_logs)
    overall_acc = sum(log["hit"] for log in all_logs) / total if total > 0 else 0
    
    print(f"\n{'='*60}")
    print(f"FINAL RESULTS")
    print(f"{'='*60}")
    print(f"Overall accuracy  : {overall_acc:.4f}")
    
    for category in categories:
        cat_logs = [log for log in all_logs if log["category"] == category]
        cat_acc = sum(log["hit"] for log in cat_logs) / len(cat_logs) if cat_logs else 0
        print(f"  - {category}: {cat_acc:.4f}")
    
    print(f"{'='*60}")
    print(f"Saved Excel  → {excel_path}")
    print(f"Saved JSON   → {json_path}")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/pcd_kaggle_runner.py
"""Self-contained Kaggle orchestration; ML dependencies load only inside GPU workers."""

import base64
import csv
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import time


CATEGORIES = ("orientation_on_the_left", "orientation_in_front_of")
LABELS = ("A", "B", "C", "D")
EXPORT_COLUMNS = [
    "index", "qid", "question", "A", "B", "C", "D", "prediction",
    "category", "col9", "col10", "answer", "hit",
]


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def circular_id(qid):
    # Match the repo scorer: remove the option-rotation suffix, retain '-flip'.
    return re.sub(r"-\d$", "", str(qid))


def get_options(row):
    return {label: str(row.get(label, "")).strip() for label in LABELS
            if str(row.get(label, "")).strip()}


def read_benchmark(tsv_path):
    """Read the official circular TSV, including index-based image references."""
    csv.field_size_limit(2**30)
    images, rows, seen = {}, [], set()
    with Path(tsv_path).open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream, delimiter="\t")
        required = {"index", "qid", "question", "answer", "category", "image", "A", "B"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f"TSV is missing columns: {required - set(reader.fieldnames or [])}")
        for raw in reader:
            index = raw["index"].strip()
            if not index or index in seen:
                raise ValueError(f"Missing/duplicate TSV index: {index!r}")
            seen.add(index)
            images[index] = raw["image"].strip()
            if raw["category"] not in CATEGORIES:
                continue
            row = {key: raw.get(key, "").strip() for key in
                   ("qid", "question", "answer", "category", *LABELS)}
            row["row_id"] = index
            row["base_qid"] = circular_id(row["qid"])
            options = get_options(row)
            if len(options) < 2 or row["answer"] not in options:
                raise ValueError(f"Invalid options/answer for {row['qid']}")
            rows.append(row)
    if not rows:
        raise ValueError("No supported allocentric rows. Use the official circular TSV.")
    if len({row["qid"] for row in rows}) != len(rows):
        raise ValueError("Duplicate qid in the selected benchmark.")
    return rows, images


def resolve_image(index, images):
    seen = set()
    while True:
        if index in seen:
            raise ValueError(f"Cyclic image reference: {index}")
        seen.add(index)
        value = images[index]
        if value in images:
            index = value
            continue
        if not value:
            raise ValueError(f"Empty image for index {index}")
        if value.startswith("data:image/"):
            value = value.split(",", 1)[1]
        # Reject a short unresolved numeric index rather than base64-decoding it.
        if value.isdigit() and len(value) < 32:
            raise ValueError(f"Missing image reference: {value}")
        try:
            return base64.b64decode(value, validate=True)
        except (ValueError, base64.binascii.Error) as error:
            raise ValueError(f"Invalid embedded image for index {index}") from error


def validate_circular_groups(rows):
    groups = {}
    for row in rows:
        groups.setdefault(row["base_qid"], []).append(row)
    for qid, group in groups.items():
        first_options = get_options(group[0])
        expected = len(first_options)
        signatures = {tuple(get_options(row).values()) for row in group}
        answers = {get_options(row)[row["answer"]] for row in group}
        if (len(group) != expected or len(signatures) != expected or len(answers) != 1
                or len({row["category"] for row in group}) != 1
                or any(sorted(get_options(row).values()) != sorted(first_options.values())
                       for row in group)):
            raise ValueError(f"Incomplete/inconsistent circular group {qid}: {len(group)} rows, "
                             f"expected {expected}. Use the circular TSV without row limits.")
    return groups


def model_signature(model_dir):
    model_dir = Path(model_dir)
    weights = sorted(model_dir.glob("*.safetensors"))
    if not weights:
        raise ValueError(f"No safetensors weights in {model_dir}")
    files = sorted({*weights, *model_dir.glob("*.json"), *model_dir.glob("*.txt"),
                    *model_dir.glob("*.model"), *model_dir.glob("*.jinja")})
    # Content hashes also protect resume when a mounted local model changes.
    return {path.name: sha256_file(path) for path in files}


def runtime_versions():
    versions = {}
    for package in ("torch", "torchvision", "transformers", "accelerate", "pandas", "Pillow",
                    "tokenizers", "huggingface_hub", "safetensors"):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    return versions


def prepare_run(config):
    root = Path(config["output_dir"])
    root.mkdir(parents=True, exist_ok=True)
    image_dir = root / "images"
    image_dir.mkdir(exist_ok=True)
    all_rows, images = read_benchmark(config["data_path"])
    validate_circular_groups(all_rows)
    print("Allocentric rows (all rotations and flips):", len(all_rows))
    for category in CATEGORIES:
        subset = [row for row in all_rows if row["category"] == category]
        print(f"  {category}: {len(subset)} rows, "
              f"{len({row['base_qid'] for row in subset})} circular groups")
    if config["mode"] == "sanity":
        selected_qid = config.get("sanity_qid")
        rows = [row for row in all_rows if row["qid"] == selected_qid] if selected_qid else all_rows[:1]
        if len(rows) != 1:
            raise ValueError(f"Sanity qid must select exactly one row: {selected_qid!r}")
    else:
        rows = [r for r in all_rows if r["qid"] in config["replay_ids"]]
        assert len(rows)==len(config["replay_ids"])
    for row in rows:
        payload = resolve_image(row["row_id"], images)
        image_hash = hashlib.sha256(payload).hexdigest()
        image_path = image_dir / f"{image_hash}.img"
        if not image_path.exists():
            image_path.write_bytes(payload)
        row["image_path"] = str(image_path)
        row["image_sha256"] = image_hash
    del images
    groups = sorted({row["base_qid"] for row in rows})
    owners = {qid: index % config["num_workers"] for index, qid in enumerate(groups)}
    manifest = {
        "schema": 1, "mode": config["mode"], "categories": list(CATEGORIES),
        "alpha": config["alpha"], "max_pixels": config["max_pixels"],
        "min_pixels": config["min_pixels"], "compute_baseline": config["compute_baseline"],
        "num_workers": config["num_workers"], "seed": config["seed"],
        "dtype": "float16", "attention": "sdpa", "transformers": "4.51.3",
        "runtime_versions": runtime_versions(),
        "dataset_sha256": sha256_file(config["data_path"]),
        "model_files": model_signature(config["model_dir"]),
        "reference_sha256": sha256_file(config["reference_path"]),
        "runner_sha256": sha256_file(__file__),
        "rows": [{key: row[key] for key in ("row_id", "qid", "answer", "image_sha256")}
                 for row in rows],
    }
    # Paths and download locations do not change the scientific run identity.
    manifest["run_fingerprint"] = fingerprint(manifest)
    old_manifest = root / "run_manifest.json"
    if config.get("resume_from") and not old_manifest.exists():
        previous = Path(config["resume_from"])
        previous_manifest = read_json(previous / "run_manifest.json")
        if previous_manifest["run_fingerprint"] != manifest["run_fingerprint"]:
            raise ValueError("Resume manifest differs. Match the model, data, code, and settings.")
        for rank in range(config["num_workers"]):
            source = previous / f"predictions_gpu{rank}.jsonl"
            destination = root / source.name
            if source.exists():
                if destination.exists():
                    if sha256_file(source) == sha256_file(destination):
                        continue
                    raise ValueError(f"Refusing to overwrite existing checkpoint: {destination}")
                shutil.copy2(source, destination)
    if old_manifest.exists():
        if read_json(old_manifest)["run_fingerprint"] != manifest["run_fingerprint"]:
            raise ValueError("Existing run uses different settings. Choose a new OUTPUT_DIR.")
    elif any(root.glob("predictions_gpu*.jsonl")) and not config.get("resume_from"):
        raise ValueError("Predictions exist without a manifest; choose a new OUTPUT_DIR.")
    write_json(old_manifest, manifest)
    write_json(root / "config.json", config)
    for rank in range(config["num_workers"]):
        shard = rows if config["mode"] == "sanity" else [
            row for row in rows if owners[row["base_qid"]] == rank]
        write_json(root / f"plan_gpu{rank}.json", shard)
    print("Run fingerprint:", manifest["run_fingerprint"])
    return rows


def read_checkpoint(path, repair_tail=False):
    """Keep completed rows; only an interrupted, unterminated tail may be repaired."""
    path = Path(path)
    if not path.exists():
        return []
    records, end = [], 0
    with path.open("rb") as stream:
        while line := stream.readline():
            if not line.endswith(b"\n"):
                if not repair_tail:
                    raise ValueError(f"Interrupted final checkpoint line: {path}")
                with path.open("r+b") as writable:
                    writable.truncate(end)
                break
            try:
                records.append(json.loads(line))
            except (json.JSONDecodeError, UnicodeDecodeError) as error:
                raise ValueError(f"Corrupt checkpoint at byte {end}: {path}") from error
            end = stream.tell()
    if len({row["row_id"] for row in records}) != len(records):
        raise ValueError(f"Duplicate row_id in checkpoint: {path}")
    return records


def validate_records(rows, records, run_fingerprint):
    expected = {row["row_id"]: row for row in rows}
    for record in records:
        row = expected.get(record["row_id"])
        if row is None or record.get("run_fingerprint") != run_fingerprint:
            raise ValueError("Checkpoint row belongs to a different run/shard.")
        if any(record.get(key) != row[key] for key in ("qid", "answer", "category")):
            raise ValueError("Checkpoint question/answer/category differs from the plan.")
        for prefix in ("", "baseline_"):
            prediction = record.get(prefix + "prediction")
            if prediction is None and prefix:
                continue
            if (prediction not in get_options(row)
                    or record.get(prefix + "hit") != int(prediction == row["answer"])):
                raise ValueError(f"Invalid stored prediction/hit for {row['qid']}")


def parse_letter(text, labels):
    # Deliberately match process_category() in the reference implementation.
    letter = text.strip()[0].upper() if text.strip() else "A"
    fallback = letter not in labels
    return (labels[0] if fallback else letter), fallback or not text.strip()


def run_worker(config_path, rank):
    import importlib.util
    import torch
    import transformers
    from PIL import Image
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

    config = read_json(config_path)
    root = Path(config["output_dir"])
    manifest = read_json(root / "run_manifest.json")
    if (sha256_file(__file__) != manifest["runner_sha256"]
            or sha256_file(config["reference_path"]) != manifest["reference_sha256"]
            or runtime_versions() != manifest["runtime_versions"]):
        raise RuntimeError("Code/runtime changed after preparation; prepare again in a new OUTPUT_DIR.")
    rows = read_json(root / f"plan_gpu{rank}.json")
    checkpoint = root / f"predictions_gpu{rank}.jsonl"
    completed = read_checkpoint(checkpoint, repair_tail=True)
    validate_records(rows, completed, manifest["run_fingerprint"])
    done = {record["row_id"] for record in completed}
    pending = [row for row in rows if row["row_id"] not in done]
    print(f"GPU worker {rank}: {len(done)}/{len(rows)} cached; {len(pending)} pending", flush=True)
    if not pending:
        return
    if transformers.__version__ != "4.51.3":
        raise RuntimeError("Install the pinned transformers version and restart the session.")
    if torch.cuda.device_count() != 1:
        raise RuntimeError("Each worker must see exactly one GPU via CUDA_VISIBLE_DEVICES.")
    torch.cuda.set_device(0)
    torch.set_num_threads(2)
    torch.manual_seed(config["seed"])
    torch.cuda.manual_seed_all(config["seed"])
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    device = torch.device("cuda:0")
    print(f"Physical GPU {rank}: {torch.cuda.get_device_name(0)}; torch={torch.__version__}", flush=True)
    spec = importlib.util.spec_from_file_location("pcd_reference", config["reference_path"])
    reference = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(reference)
    processor_kwargs = {key: config[key] for key in ("min_pixels", "max_pixels")
                        if config[key] is not None}
    processor = AutoProcessor.from_pretrained(config["model_dir"], local_files_only=True,
                                             **processor_kwargs)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        config["model_dir"], torch_dtype=torch.float16, device_map={"": 0},
        attn_implementation="sdpa", local_files_only=True, low_cpu_mem_usage=True,
    ).eval()
    token_ids = {label: reference.get_candidate_token_ids_for_label(label, processor.tokenizer)
                 for label in LABELS}
    # Extraction is text-only; viewpoint cache keys include the actual (possibly flipped) image.
    extraction_cache, viewpoint_cache = {}, {}
    for record in completed:
        extraction_cache[(record["category"], record["question"])] = (
            record["ref_obj"], record["target_obj"])
        viewpoint_cache[(record["category"], record["image_sha256"], record["ref_obj"])] = (
            record["viewpoint_raw"], record["is_same_viewpoint"])
    with checkpoint.open("a", encoding="utf-8") as sink, torch.inference_mode():
        for ordinal, row in enumerate(pending, 1):
            start = time.monotonic()
            try:
                torch.cuda.reset_peak_memory_stats()
                with Image.open(row["image_path"]) as original_image:
                    image = original_image.convert("RGB")
                category, question = row["category"], row["question"]
                options = get_options(row)
                labels = list(options)
                extraction_key = (category, question)
                if extraction_key not in extraction_cache:
                    extraction_cache[extraction_key] = reference.extract_objects_via_llm(
                        question, category, model, processor, device)
                ref_obj, target_obj = extraction_cache[extraction_key]
                rewritten = reference.build_rewrite_q(ref_obj, target_obj, category)
                options_text = reference.options_to_text(options)
                original_prompt = reference.build_original_prompt(question, options_text)
                camera_prompt = reference.build_camera_prompt(rewritten, options_text)
                viewpoint_key = (category, row["image_sha256"], ref_obj)
                if viewpoint_key not in viewpoint_cache:
                    viewpoint_cache[viewpoint_key] = reference.check_same_viewpoint(
                        image, ref_obj, category, model, processor, device)
                viewpoint_raw, is_same = viewpoint_cache[viewpoint_key]
                scores, camera_raw, fallback = None, None, False
                if is_same:
                    camera_raw = reference.generate_answer(image, camera_prompt, model, processor, device)
                    prediction, fallback = parse_letter(camera_raw, labels)
                else:
                    prediction, scores = reference.contrastive_pick_option(
                        image, original_prompt, camera_prompt, labels, config["alpha"],
                        model, processor, device, token_ids)
                if scores is None:
                    _, scores = reference.contrastive_pick_option(
                        image, original_prompt, camera_prompt, labels, config["alpha"],
                        model, processor, device, token_ids)
                if scores is not None and not all(
                        math.isfinite(value) for candidate in scores.values() for value in candidate.values()):
                    raise FloatingPointError("Non-finite option scores; inspect model/dtype before resuming.")
                record = {**row, "run_fingerprint": manifest["run_fingerprint"],
                          "gpu_rank": rank, "ref_obj": ref_obj, "target_obj": target_obj,
                          "extraction_fallback": (ref_obj, target_obj) == ("object", "object"),
                          "rewrite_question": rewritten, "viewpoint_raw": viewpoint_raw,
                          "is_same_viewpoint": is_same, "branch": "camera_direct" if is_same else "pcd",
                          "camera_raw": camera_raw, "answer_parse_fallback": fallback,
                          "prediction": prediction, "hit": int(prediction == row["answer"]),
                          "scores": scores}
                if config["compute_baseline"]:
                    baseline_raw = reference.generate_answer(image, original_prompt, model, processor, device)
                    baseline, baseline_fallback = parse_letter(baseline_raw, labels)
                    record.update(baseline_raw=baseline_raw, baseline_prediction=baseline,
                                  baseline_hit=int(baseline == row["answer"]),
                                  baseline_parse_fallback=baseline_fallback)
                if True:
                    record.update(original_prompt=original_prompt, camera_prompt=camera_prompt,
                                  viewpoint_prompt=reference.build_viewpoint_check_prompt(ref_obj, category),
                                  option_token_ids=token_ids, image_size=list(image.size))
                if camera_raw is None:
                    camera_raw = reference.generate_answer(image,camera_prompt,model,processor,device)
                record["diagnostic_camera_prediction"]=parse_letter(camera_raw,labels)[0]
                record["diagnostic_camera_raw"]=camera_raw
                record["diagnostic_contrastive_prediction"]=max(labels,key=lambda label:scores[label]["contrastive"])
                record["extraction_prompt"],record["extraction_raw"]=reference.extraction_trace[extraction_key]
                torch.cuda.synchronize()
                record["seconds"] = time.monotonic() - start
                record["peak_vram_gib"] = torch.cuda.max_memory_allocated() / 2**30
                sink.write(json.dumps(record, ensure_ascii=False) + "\n")
                sink.flush()
                os.fsync(sink.fileno())
                print(f"[{len(done) + ordinal}/{len(rows)}] {row['qid']} "
                      f"pred={prediction} answer={row['answer']} branch={record['branch']} "
                      f"time={record['seconds']:.2f}s vram={record['peak_vram_gib']:.2f}GiB", flush=True)
            except Exception as error:
                write_json(root / f"error_gpu{rank}.json", {
                    "row_id": row["row_id"], "qid": row["qid"], "error": repr(error),
                    "hint": "Fix the error, then rerun. Completed JSONL rows are preserved. "
                            "If changing model/settings, use a new OUTPUT_DIR.",
                })
                raise


def launch_workers(config):
    root = Path(config["output_dir"])
    processes, streams = [], []
    try:
        for rank in range(config["num_workers"]):
            log_path = root / f"worker_gpu{rank}.log"
            stream = log_path.open("w", encoding="utf-8")
            streams.append(stream)
            environment = dict(os.environ, CUDA_VISIBLE_DEVICES=str(rank),
                               TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS="2",
                               PYTHONUNBUFFERED="1")
            process = subprocess.Popen(
                [sys.executable, "-u", str(Path(__file__).resolve()), "--worker",
                 str(root / "config.json"), str(rank)], env=environment,
                stdout=stream, stderr=subprocess.STDOUT,
            )
            processes.append(process)
        while True:
            statuses = [process.poll() for process in processes]
            for rank, status in enumerate(statuses):
                if status not in (None, 0):
                    tail = "\n".join((root / f"worker_gpu{rank}.log").read_text(
                        encoding="utf-8", errors="replace").splitlines()[-30:])
                    raise RuntimeError(f"GPU {rank} exited with status {status}:\n{tail}")
            for rank in range(config["num_workers"]):
                log = (root / f"worker_gpu{rank}.log").read_text(encoding="utf-8", errors="replace")
                print(f"GPU {rank}: " + (log.splitlines()[-1] if log.splitlines() else "loading..."), flush=True)
            if all(status == 0 for status in statuses):
                break
            time.sleep(15)
    finally:
        for process in processes:
            if process.poll() is None:
                process.terminate()
        for process in processes:
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        for stream in streams:
            stream.close()


def compute_metrics(rows, records, include_baseline=True):
    groups = validate_circular_groups(rows)
    if len({record["row_id"] for record in records}) != len(records):
        raise ValueError("Duplicate predictions across shards.")
    by_id = {record["row_id"]: record for record in records}
    expected = {row["row_id"] for row in rows}
    if set(by_id) != expected:
        raise ValueError(f"Incomplete run: {len(expected - set(by_id))} missing, "
                         f"{len(set(by_id) - expected)} unexpected rows. Rerun workers to resume.")
    report = []
    for method, hit_key in [("pcd", "hit")] + ([("baseline", "baseline_hit")] if include_baseline else []):
        for category in ("overall", *CATEGORIES):
            selected = [row for row in rows if category == "overall" or row["category"] == category]
            selected_groups = [group for group in groups.values()
                               if category == "overall" or group[0]["category"] == category]
            correct = sum(all(by_id[row["row_id"]][hit_key] == 1 for row in group)
                          for group in selected_groups)
            report.append({"method": method, "category": category,
                           "rows": len(selected), "circular_groups": len(selected_groups),
                           "row_accuracy_pct": 100 * sum(by_id[row["row_id"]][hit_key]
                                                          for row in selected) / len(selected) if selected else None,
                           "circular_accuracy_pct": 100 * correct / len(selected_groups) if selected_groups else None})
    return report


def export_results(config):
    import pandas as pd

    root = Path(config["output_dir"])
    run_fingerprint = read_json(root / "run_manifest.json")["run_fingerprint"]
    all_records, rows = [], []
    for rank in range(config["num_workers"]):
        plan = read_json(root / f"plan_gpu{rank}.json")
        records = read_checkpoint(root / f"predictions_gpu{rank}.jsonl")
        validate_records(plan, records, run_fingerprint)
        if len(records) != len(plan):
            raise ValueError(f"GPU {rank} has {len(records)}/{len(plan)} rows; rerun workers.")
        all_records.extend(records)
        if config["mode"] != "sanity" or rank == 0:
            rows.extend(plan)
    if config["mode"] == "sanity":
        write_json(root / "sanity_results.json", all_records)
        return pd.DataFrame([{key: record.get(key) for key in (
            "gpu_rank", "qid", "branch", "prediction", "answer", "hit", "baseline_prediction",
            "viewpoint_raw", "extraction_fallback", "seconds", "peak_vram_gib")}
                             for record in all_records]), all_records
    metrics = compute_metrics(rows, all_records, config["compute_baseline"])
    metric_frame = pd.DataFrame(metrics)
    metric_frame.to_csv(root / "metrics.csv", index=False)
    write_json(root / "metrics.json", metrics)
    all_records.sort(key=lambda row: int(row["row_id"]))
    write_json(root / "predictions.json", all_records)
    pd.DataFrame(all_records).to_csv(root / "predictions.csv", index=False)
    alpha = str(config["alpha"]).replace(".", "p")
    model_name = f"PCD_Qwen2.5-VL-3B-Instruct_alpha{alpha}"
    export_dir = root / "outputs" / model_name
    export_dir.mkdir(parents=True, exist_ok=True)
    excel_rows = []
    for index, row in enumerate(all_records):
        excel_rows.append([index, row["qid"], row["question"], *[row[key] for key in LABELS],
                           row["prediction"], row["category"], row["rewrite_question"],
                           row["prediction"], row["answer"], row["hit"]])
    pd.DataFrame(excel_rows, columns=EXPORT_COLUMNS).to_excel(
        export_dir / f"{model_name}_3DSRBenchv1_openai_result.xlsx", index=False)
    write_json(export_dir / f"{model_name}_results.json", all_records)
    write_json(root / "diagnostics.json", {
        "rows": len(all_records),
        "extraction_fallback_rows": sum(row["extraction_fallback"] for row in all_records),
        "answer_parse_fallback_rows": sum(row["answer_parse_fallback"] for row in all_records),
        "camera_direct_rows": sum(row["is_same_viewpoint"] for row in all_records),
        "pcd_rows": sum(not row["is_same_viewpoint"] for row in all_records),
    })
    return metric_frame, all_records


if __name__ == "__main__":
    if len(sys.argv) != 4 or sys.argv[1] != "--worker":
        raise SystemExit("Usage: kaggle_runner.py --worker CONFIG_JSON RANK")
    run_worker(sys.argv[2], int(sys.argv[3]))

## 5. Lập danh sách chạy

Kiểm tra đầy đủ các hoán vị trước khi chạy. Ảnh base64 và ảnh tham chiếu theo index đều
được giải quyết trước khi chia shard. Hash model/data/code/settings ngăn ghép nhầm checkpoint.
Cell này đọc toàn bộ TSV và hash weights một lần; hai worker chỉ đọc shard đã chuẩn bị.

In [ ]:
import importlib.util

runner_path = Path("/kaggle/working/pcd_kaggle_runner.py")
reference_path = Path("/kaggle/working/pcd_reference.py")
spec = importlib.util.spec_from_file_location("pcd_kaggle_runner", runner_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
CONFIG = {
    "mode": MODE, "model_dir": str(model_path), "data_path": str(Path(TSV_PATH).resolve()),
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "dataset_revision": DATASET_REVISION,
    "output_dir": str(OUTPUT_DIR), "reference_path": str(reference_path),
    "alpha": float(ALPHA), "num_workers": NUM_WORKERS, "seed": SEED,
    "compute_baseline": COMPUTE_BASELINE, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS,
    "sanity_qid": SANITY_QID, "resume_from": str(RESUME_FROM) if RESUME_FROM else None,
}
CONFIG["replay_ids"] = sorted(OLD)
selected_rows = runner.prepare_run(CONFIG)
print("Selected rows:", len(selected_rows))
for rank in range(NUM_WORKERS):
    plan = runner.read_json(OUTPUT_DIR / f"plan_gpu{rank}.json")
    print(f"GPU {rank}: {len(plan)} rows")

## 6. Chạy đồng thời trên hai GPU

Tiến độ in mỗi 15 giây; log chi tiết nằm trong `worker_gpu*.log`.
Mỗi dòng hoàn thành được flush/fsync vào `predictions_gpu*.jsonl`.
Nếu hết phiên hoặc lỗi: giữ output, sửa nguyên nhân rồi chạy lại cell này để resume.
Lỗi sẽ dừng thí nghiệm; không tự bỏ dòng hoặc giảm độ phân giải làm lệch phép so sánh.

In [ ]:
# Separate Python subprocesses avoid notebook multiprocessing/fork/CUDA problems.
# On failure, the full traceback is in worker_gpu0.log / worker_gpu1.log.
# Rerun THIS cell to continue from the last completed JSONL row.
runner.launch_workers(CONFIG)

## 7. Chấm CircularEval và xuất kết quả

Một nhóm chỉ đúng khi **tất cả** hoán vị đáp án đều đúng. Ảnh `-flip` là nhóm riêng, đúng như scorer của repo. Thiếu dòng hoặc trùng dòng sẽ báo lỗi trước khi xuất điểm.

In [ ]:
from IPython.display import display

metrics, records = runner.export_results(CONFIG)
display(metrics)
display(runner.read_json(OUTPUT_DIR / "diagnostics.json"))
print("Đọc circular_accuracy_pct để so với paper; row_accuracy_pct là số phụ trợ.")
print("Paper, Bảng 2/5: PCD 3B α=4 → overall 58.73%, front/behind 49.13%, left/right 68.19%.")
print("Baseline tham khảo trong paper: 38.67%, 42.73%, 34.67%; không phải ngưỡng pass/fail.")
print("Notebook giữ prompt/code repo; GPU, FP16/SDPA và thư viện có thể làm kết quả khác paper.")
print("Metrics:", OUTPUT_DIR / "metrics.csv")

## Lưu kết quả

ZIP chứa manifest, checkpoint từng GPU, log và kết quả; không chứa weights/ảnh.

In [ ]:
(OUTPUT_DIR/'old_vs_replay.json').write_text(json.dumps([dict(old=OLD[r['qid']],replay_prediction=r['prediction'],replay_branch=r['branch']) for r in records]))
import zipfile

# Save small reproducible outputs; exclude image payloads and downloaded model weights.
archive = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_results.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            bundle.write(path, Path(OUTPUT_DIR.name) / path.relative_to(OUTPUT_DIR))
    bundle.write(reference_path, "pcd_reference.py")
    bundle.write(runner_path, "pcd_kaggle_runner.py")
print("Kaggle Output ZIP:", archive)
print("Dùng Save Version → Save & Run All để lưu output qua phiên Kaggle.")

### Chạy tiếp ở phiên Kaggle mới

1. Lưu Output của phiên cũ, gắn nó vào notebook mới bằng **Add Input**.
2. Đặt `RESUME_FROM` tới thư mục có `run_manifest.json` và `predictions_gpu0/1.jsonl`.
   Giữ nguyên α, độ phân giải, baseline và mã notebook. Có thể dùng model/TSV đã gắn để giảm tải.
3. Run All. Các dòng đã ghi hoàn chỉnh được giữ; dòng cuối bị cắt do ngắt phiên sẽ được chạy lại.

Nếu CUDA OOM, xem log để biết bước lỗi. Có thể thử `MAX_PIXELS = 1280 * 28 * 28`
trong **OUTPUT_DIR mới** và ghi rõ thay đổi khi so với paper. FP16 phù hợp T4;
không bật BF16 hoặc FlashAttention 2 trong notebook này.